# Apply an HSC reference pipeline

This path needs fitted model bundles and your PDZs/observables, not Optuna or retraining. The example adapter reads existing HSC compact products; replace it with your own `PDZDataset` adapter for another input format. A different survey requires its own trained models.


In [ ]:
MODEL_DIR = "artifacts/hsc-pdr3/dnnz-dud"
DATA_DIR = "data/hsc-pdr3/dnnz_dud"
OUTPUT_FILE = "runs/hsc-predictions.csv"
BATCH_SIZE = 4096
APPLY_FLAGS = False  # true only when the input supplies the required grizy photometry


In [ ]:
from pathlib import Path
from turbopdz import TurboPDZ
from turbopdz.recipes.hsc import read_products

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
pipeline = TurboPDZ.load(PROJECT_ROOT / MODEL_DIR)
data = read_products(PROJECT_ROOT / DATA_DIR)
result = pipeline.predict(data.pdz, data.z_grid, data.observables,
                          object_id=data.object_id, batch_size=BATCH_SIZE, applicability=APPLY_FLAGS)
output = PROJECT_ROOT / OUTPUT_FILE
output.parent.mkdir(parents=True, exist_ok=True)
if output.exists():
    raise FileExistsError("Choose a new output file; existing predictions are never overwritten.")
result.to_csv(output, index=False)
result.head()
